# Final Project!

Author: Cameron Mullaney

This project is split into 2 major parts: 
1. Building a Linear Regression Model
2. Using the model to process streaming data

This will all be done using `pyspark` primarily, with pandas used as well. 

In [1]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from Proj2Script import *

## Pyspark MLlib-specific tools for model evaluation
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, VectorIndexer, OneHotEncoder, PCA
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator 

## Part I - Fitting the Model

Part I is comprised of 3 major tasks:

1. Creating a data transformation pipeline
2. Using that pipeline to fit a LR model
3. Evaluating the LR model.

Starting the spark session

In [2]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 19:38:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading the data in as a pandas DF, then converting to spark DF

In [3]:
power = pd.read_csv("power_ml_data.csv")
power = spark.createDataFrame(power) 
power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Looks good! Let's get to transforming

### Building the Pipeline

This will be a pretty complicated pipeline, containing many transformations.

For `hour`, I am casting it to a double, and then binarizing it with a cutoff of 6.5

In [4]:
hour_double = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

In [5]:
hour_bin = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "Hour_b")

For `month`, I am casting it to a double so I can One-hot encode it

In [6]:
month_double = SQLTransformer(
    statement="SELECT *, CAST(Month AS DOUBLE) AS Month_d FROM __THIS__"
)

In [7]:
month_ohe = OneHotEncoder(inputCol = "Month_d", outputCol = "Month_ohe")

Here I'm creating my `label` column

In [8]:
label_cast = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

Here I'm creating the list of variables I will send to the PCA

In [9]:
PCA_assembler = VectorAssembler(inputCols = ["Temperature", 
                                         "Humidity", 
                                         "Wind_Speed", 
                                         "General_Diffuse_Flows",
                                         "Diffuse_Flows"], 
                            outputCol = "pca_input", 
                            handleInvalid = 'keep')

In [10]:
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features")

And here I'm running the PCA_assembler to fit the PCA.

In [11]:
temp = PCA_assembler.transform(
    label_cast.transform(power))

pca_fit = pca.fit(temp)        

This is the final assembler, creating the final "features" column for later fitting.

In [12]:
assembler = VectorAssembler(inputCols = ["pca_features", 
                                         "Hour_b", 
                                         "Power_Zone_1", 
                                         "Power_Zone_2",
                                         "Month_ohe"], 
                            outputCol = "features", 
                            handleInvalid = 'keep')

Time to test-run the transformations:

In [13]:
df = hour_double.transform(power)
df = hour_bin.transform(df)
df = month_double.transform(df)
df = month_ohe.fit(df).transform(df)
df = PCA_assembler.transform(df)
df = pca_fit.transform(df)
df = label_cast.transform(df)
temp = assembler.transform(df)

In [14]:
temp.show(5, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_d|Hour_b|Month_d|Month_ohe     |pca_input                     |pca_features                            |label      |features                                                                             |
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+------+------+-------+--------------+------------------------------+----------------------------------------+-----------+-------------------------------------------------------------------------------------+
|6.559      |73.8  

This looks good! Time to create our Pipeline

First I'll define what kind of model we want

In [15]:
lr = LinearRegression(featuresCol = "features", labelCol = "label")

In [16]:
pipe_1 = Pipeline(stages = [hour_double, hour_bin, month_double, month_ohe, PCA_assembler, pca, label_cast, assembler, lr])

### Building and fitting the model

Creating our list of potential parameters

In [17]:
LR_paramGrid = ParamGridBuilder()\
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])\
    .build()


In [18]:
LRCV = CrossValidator(estimator = pipe_1,
                      estimatorParamMaps = LR_paramGrid,
                      evaluator = RegressionEvaluator(metricName = "rmse"),
                      numFolds=5,
                      parallelism = 4,
                      seed = 12)

This is going to take a while - and produce many warnings.

In [19]:
LRCV_model = LRCV.fit(power)

26/04/29 19:38:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/29 19:38:17 WARN Instrumentation: [74f31fe4] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [990f455c] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [e0df70f6] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:17 WARN Instrumentation: [43426161] regParam is zero, which might cause numerical instability and overfitting.
26/04/29 19:38:19 WARN Instrumentation: [74f31fe4] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 19:38:19 WARN Instrumentation: [990f455c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/29 19:38:19 W

### Reporting model values

Okay! Let's see what our optimal values are

In [30]:
LR_list = []
for i in range(len(LR_paramGrid)):
    LR_list.append([LRCV_model.avgMetrics[i], LR_paramGrid[i].values()])
LR_list.sort(key=lambda x: x[0])
print("CV Error \t\t\t [regParam, ElasticParam]")
LR_list[0]

CV Error 			 [regParam, ElasticParam]


[2147.879982678048, dict_values([0.1, 0.1])]

Training RMSE:

In [31]:
LR_pred = LRCV_model.transform(power)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(LR_pred)
print("Training RMSE: \t", training_rmse)

Training RMSE: 	 2147.097320321586


Residuals:

In [32]:
resid_df = LR_pred.withColumn("residual", col("label") - col("prediction"))
resid_df.select("residual", "label", "prediction").show()

+------------------+-----------+------------------+
|          residual|      label|        prediction|
+------------------+-----------+------------------+
|-638.6844049039391|20240.96386| 20879.64826490394|
|1471.1367385456979|20131.08434|18659.947601454303|
|1463.9585040226702|19668.43373| 18204.47522597733|
|1308.8696312355132|18899.27711|17590.407478764486|
|1445.3432101497056|18442.40964|16997.066429850296|
|1612.6517153933491|18130.12048|16517.468764606652|
|1852.0122508872864|17945.06024|16093.047989112712|
|1736.7707088119987|17459.27711|   15722.506401188|
|1754.6696976219446|17025.54217|15270.872472378056|
|1856.0333437770769|16794.21687|14938.183526222923|
| 1985.793232231279|16638.07229|14652.279057768721|
| 1980.376702661104|16395.18072|14414.804017338896|
|2034.8443820286411|16117.59036|14082.745977971359|
|2197.8897606052833| 15822.6506|13624.760839394718|
| 2222.036787445266|15672.28916|13450.252372554734|
| 2294.906820974009|15597.10843|13302.201609025991|
| 2355.57556